### 1. Análisis Exploratorio y Preprocesamiento Inicial

En esta primera fase del proyecto, llevamos a cabo la ingesta de datos y el acondicionamiento del texto para su posterior análisis algorítmico. Este proceso se divide en tres etapas fundamentales:

1. **Carga de Datos:** Utilizamos la biblioteca `pandas` para importar la base de datos cruda desde nuestro repositorio local.
2. **Análisis Exploratorio de Datos (EDA):** Extraemos las métricas estructurales del conjunto de datos. Esto incluye conocer el volumen total de registros, la distribución de las clases de sentimiento (para detectar posibles desbalanceos que afecten a los modelos predictivos) y la identificación de valores nulos o ausentes.
3. **Limpieza de Texto (NLP Básico):** Implementamos una función basada en expresiones regulares (`re`) para reducir el ruido en los mensajes. Se eliminan enlaces web (URLs) y menciones directas a usuarios, ya que estos elementos aumentan la dimensionalidad del vocabulario sin aportar carga emocional o semántica útil. Por último, se aplica un proceso de normalización convirtiendo todos los caracteres a minúsculas.

In [ ]:
import pandas as pd
import re

# 1. Carga de datos
ruta_archivo = '../data/raw/fifa_world_cup_2022_tweets.csv'
df = pd.read_csv(ruta_archivo)

# 2. Exploración inicial
print("--- DIMENSIONES DEL DATASET ---")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

print("\n--- DISTRIBUCIÓN DE SENTIMIENTOS ORIGINAL ---")
print(df['Sentiment'].value_counts(dropna=False))

print("\n--- VALORES NULOS POR COLUMNA ---")
print(df.isnull().sum())

# 3. Función de limpieza de texto
def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    # Eliminar URLs
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    # Eliminar menciones a usuarios
    texto = re.sub(r'\@\w+', '', texto)
    # Convertir a minúsculas y eliminar espacios extra
    texto = texto.lower().strip()
    return texto

# Aplicar limpieza
df['Tweet_Limpio'] = df['Tweet'].apply(limpiar_texto)

# Ver el resultado de la limpieza
print("\n--- MUESTRA DE TEXTO LIMPIO ---")
print(df[['Tweet', 'Tweet_Limpio']].head(3))

--- DATOS TEMPORALES ---
Tabla de evolución temporal guardada con éxito.

--- ANÁLISIS POR ACTOR/TEMA ---
  Actor_Tema Sentimiento  Porcentaje  Volumen_Total
0       Fifa     neutral       40.19           4810
1       Fifa    positive       39.54           4810
2       Fifa    negative       20.27           4810
3      Qatar     neutral       42.11           9523
4      Qatar    positive       30.91           9523
5      Qatar    negative       26.98           9523
6      Messi    positive       57.49            327
7      Messi     neutral       38.84            327
8      Messi    negative        3.67            327
9    Ronaldo    positive       51.21            207


### 2. Extracción de Entidades y Construcción del Grafo

Una vez normalizado el texto, procedemos a la extracción de relaciones para el Análisis de Redes Sociales (ARS). Dado que el dataset proporciona el contenido del mensaje pero anonimiza al autor original, estructuraremos la red como un grafo dirigido bipartito o de afiliación:
- **Nodos de origen:** Representan cada tuit individual, etiquetados con su respectivo sentimiento (Positivo, Negativo o Neutro).
- **Nodos de destino:** Representan a los usuarios o entidades mencionadas explícitamente en el texto original (extraídos mediante expresiones regulares).
- **Aristas (Edges):** Representan la acción de mencionar, ponderadas por la frecuencia de interacción.

Este enfoque estructural nos permitirá exportar la topología de la red a formato GEXF. Posteriormente, mediante el software Gephi, aplicaremos algoritmos de detección de comunidades (como Louvain) para identificar si los sentimientos se polarizan en torno a actores específicos. Finalmente, el conjunto de datos procesado se exportará a un archivo tabular para alimentar el panel de control interactivo.

In [4]:
import networkx as nx

# 1. Función para extraer menciones del texto original
def extraer_menciones(texto):
    if not isinstance(texto, str):
        return []
    # Busca cualquier palabra que empiece por @
    return re.findall(r'@(\w+)', texto)

# Aplicamos la función sobre el texto crudo (antes de la limpieza)
df['Menciones'] = df['Tweet'].apply(extraer_menciones)

# 2. Construcción del Grafo Dirigido con NetworkX
G = nx.DiGraph()

print("Construyendo el grafo de interacciones...")
for index, row in df.iterrows():
    # Creamos un identificador único para el mensaje original
    origen = f"Tuit_{index}" 
    menciones = row['Menciones']
    sentimiento = row['Sentiment']
    
    # Añadimos el nodo de origen y le asignamos el sentimiento como atributo
    G.add_node(origen, tipo="Mensaje", sentimiento=sentimiento)
    
    for destino in menciones:
        # Añadimos al usuario mencionado si no existe
        if not G.has_node(destino):
            G.add_node(destino, tipo="Usuario_Mencionado", sentimiento="N/A")
            
        # Añadimos la conexión (arista)
        if G.has_edge(origen, destino):
            G[origen][destino]['weight'] += 1
        else:
            G.add_edge(origen, destino, weight=1)

# 3. Exportar el grafo a formato GEXF para Gephi
ruta_grafo = '../data/processed/red_interacciones.gexf'
nx.write_gexf(G, ruta_grafo)
print(f"Grafo exportado con éxito: {G.number_of_nodes()} nodos y {G.number_of_edges()} aristas.")
print(f"Archivo guardado en: {ruta_grafo}")

# 4. Guardar el dataset limpio para el Dashboard (Fase 4)
# Seleccionamos solo las columnas útiles y descartamos índices automáticos o listas
columnas_finales = ['Date Created', 'Number of Likes', 'Source of Tweet', 'Tweet_Limpio', 'Sentiment']
df_final = df[columnas_finales].copy()

# Renombramos temporalmente la columna de fecha para facilitar el trabajo en Dash
df_final.rename(columns={'Date Created': 'Fecha_Completa'}, inplace=True)

ruta_csv = '../data/processed/datos_limpios.csv'
df_final.to_csv(ruta_csv, index=False)
print(f"\nDataset procesado guardado con éxito. Dimensiones finales: {df_final.shape}")
print(f"Archivo guardado en: {ruta_csv}")

Construyendo el grafo de interacciones...
Grafo exportado con éxito: 25690 nodos y 6596 aristas.
Archivo guardado en: ../data/processed/red_interacciones.gexf

Dataset procesado guardado con éxito. Dimensiones finales: (22524, 5)
Archivo guardado en: ../data/processed/datos_limpios.csv


### 3. Análisis de Sentimiento Avanzado y Evolución Temporal

Para comprender la dinámica de la opinión pública, no basta con analizar la distribución global de la polaridad. Es imperativo evaluar cómo fluctúa el sentimiento a lo largo del tiempo y cómo se distribuye en torno a los principales actores o temas de debate (cruce de variables).

En esta fase, procesaremos la evolución diaria del sentimiento para identificar picos de actividad que puedan correlacionarse con eventos reales del Mundial de Qatar 2022. Asimismo, extraeremos una métrica de impacto para entidades clave (por ejemplo: la propia FIFA, el país anfitrión, jugadores destacados o controversias como el VAR), calculando la proporción de sentimiento positivo, negativo y neutro que gravita sobre cada uno de ellos. Estas tablas de datos estructurados servirán como núcleo analítico para el dashboard interactivo final.

In [1]:
import pandas as pd

# Asumimos que df_final es el dataframe que guardamos en la fase anterior
# Aseguramos el formato de fecha
df_final['Fecha_Completa'] = pd.to_datetime(df_final['Fecha_Completa'])
df_final['Fecha'] = df_final['Fecha_Completa'].dt.date

# 1. Evolución Temporal del Sentimiento
temporal_sent = df_final.groupby(['Fecha', 'Sentiment']).size().reset_index(name='counts')
temporal_sent.to_csv('../data/processed/evolucion_temporal.csv', index=False)
print("--- DATOS TEMPORALES ---")
print("Tabla de evolución temporal guardada con éxito.")

# 2. Cruce de Actores / Temas Clave (Influencers/Keywords)
# Definimos los temas o actores más relevantes del Mundial para medir su polaridad
top_targets = ['fifa', 'qatar', 'messi', 'ronaldo', 'var']
target_analysis = []

for target in top_targets:
    # Filtramos los tuits que mencionan al actor/tema en el texto limpio
    mask = df_final['Tweet_Limpio'].apply(lambda x: isinstance(x, str) and target in x)
    df_filtrado = df_final[mask]
    
    if not df_filtrado.empty:
        # Calculamos el porcentaje de cada sentimiento
        counts = df_filtrado['Sentiment'].value_counts(normalize=True) * 100
        for sent, perc in counts.items():
            target_analysis.append({
                'Actor_Tema': target.capitalize(), 
                'Sentimiento': sent, 
                'Porcentaje': round(perc, 2),
                'Volumen_Total': len(df_filtrado)
            })

df_target = pd.DataFrame(target_analysis)
df_target.to_csv('../data/processed/sentimiento_actores.csv', index=False)
print("\n--- ANÁLISIS POR ACTOR/TEMA ---")
print(df_target.head(10))

NameError: name 'df_final' is not defined